# 🧠 Trabajo de Campo – Inteligencia del Conocimiento (InCo)
## Sistema Basado en Conocimiento: Optimización de Inventarios y Reducción de Desperdicios
### 📌 Punto 2: Arquitecto de Base de Hechos (Modelado de Marcos / Frames)

---

### 🎯 Responsabilidad y Tareas del Punto 2
* **Responsabilidad:** Traducir el diccionario formal de 66 proposiciones ($A$ a $BN$) y el diseño del inventario gastronómico a código ejecutable en Python.
* **Tareas Específicas:**
  1. **Configuración del entorno:** Preparar el entorno de ejecución en Google Colab resolviendo la compatibilidad de la librería `experta` en Python 3.10+.
  2. **Modelado de Marcos (Frames):** Diseñar las clases representativas del dominio (heredando de `Fact` en Experta) según la teoría de Marcos de Marvin Minsky, encapsulando ranuras (slots) y facetas procedimentales:
     * `MarcoContexto` (alias `Contexto`): Variables ambientales, meteorológicas, calendario y demanda externa.
     * `MarcoCliente` (alias `Cliente`): Segmentación de comensales, reservas y comportamientos esperados.
     * `MarcoInsumo` (alias `Insumo`): Catálogo de ingredientes, perecibilidad, rotación y costos.
     * `MarcoProveedor` (alias `Proveedor`): Estado de comercios cercanos, carnicerías, políticas de precios.
     * `MarcoOperacion` (alias `Operacion`): Comandas activas, tareas operativas, disponibilidad del encargado y alertas de quiebre de stock.
     * `HechoSimbolico`: Representación proposicional híbrida directa ($A \dots BN$) para que los Ingenieros de Reglas (Puntos 3 y 4) programen sin fricción y con trazabilidad al 100%.
  3. **Memoria de Trabajo Inicial (WM0):** Diseñar los cargadores de hechos para los 3 Casos de Uso documentados en la Actividad 4 y un cargador interactivo.
  4. **Prueba de Inferencia:** Verificar el encadenamiento hacia adelante con una regla de prueba sobre el Caso 1.


---
## 1. Configuración del Entorno y Parche para Google Colab (Python 3.10+)

La librería `experta` requiere un ajuste de compatibilidad en entornos Python modernos donde `collections.Mapping` fue trasladado al módulo `collections.abc`. Ejecuta la siguiente celda para instalar y parchar el entorno automáticamente:


In [ ]:
# Instalación de librerías necesarias en Google Colab
!pip install frozendict experta --quiet

# Parche de compatibilidad de colecciones para Python 3.10+ en Colab
import collections
import collections.abc

for attr in [
    'Mapping', 'MutableMapping', 'Sequence', 'MutableSequence',
    'Iterable', 'MutableSet', 'Callable', 'Container'
]:
    if not hasattr(collections, attr) and hasattr(collections.abc, attr):
        setattr(collections, attr, getattr(collections.abc, attr))

from experta import Fact, Field, KnowledgeEngine, Rule, DefFacts, AS, MATCH, NOT, AND, OR
print("✅ Entorno Colab configurado con éxito. Experta lista para operar.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.66 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.
✅ Entorno Colab configurado con éxito. Experta lista para operar.


---
## 2. Modelado de Marcos de Conocimiento (Frames)

Implementamos los **Marcos (Frames)** como clases basadas en `Fact`. Cada marco representa un concepto del mundo real del restaurante con sus atributos o **Ranuras (Slots)**:


In [ ]:
# Definición de las clases de Marcos del Dominio Gastronómico

class MarcoContexto(Fact):
    """
    Marco de Conocimiento para el Entorno y Contexto del Restaurante.
    Ranuras (Slots):
      - dia: str ('lunes', ..., 'domingo', 'fin_de_semana')
      - momento: str ('manana', 'tarde', 'noche')
      - horario_critico: str ('domingo_tarde', 'ninguno')
      - temperatura: str ('frio', 'templado', 'calido')
      - clima: str ('soleado', 'lluvioso', 'nublado')
      - es_quincena: bool (True si es semana de cobro)
      - evento: str ('boca_river', 'desfile_23_agosto', 'ninguno')
      - competencia: str ('alta', 'normal', 'baja')
    """
    pass

class MarcoCliente(Fact):
    """
    Marco de Conocimiento para la Demanda y el Perfil del Comensal.
    Ranuras (Slots):
      - tipo: str ('familia', 'jovenes', 'ejecutivo', 'general')
      - reservas: str ('grupos_grandes', 'mesas_chicas', 'ninguna')
      - comportamiento: str ('alta_bebida_baja_comida', 'estandar')
      - preferencia: str ('menu_al_plato', 'bebidas_y_sandwiches')
    """
    pass

class MarcoInsumo(Fact):
    """
    Marco de Conocimiento para Insumos, Almacenamiento y Mermas.
    Ranuras (Slots):
      - nombre: str ('carne', 'cerveza', 'hielo', 'queso', etc.)
      - categoria: str ('perecedero_diario', 'bebidas_generales', 'pizza_y_aderezos')
      - estado: str ('fresco', 'fraccionada_en_folex', 'congelada', 'en_deposito')
      - apariencia_carne: str ('normal', 'color_oscuro')
      - aumento_costo_carne: bool
      - aumento_costo_bebida: bool
      - rotacion: str ('muy_rapida', 'rapida', 'normal')
      - stock_critico: bool
    """
    pass

class MarcoProveedor(Fact):
    """
    Marco de Conocimiento para Proveedores y Comercios de Abastecimiento.
    Ranuras (Slots):
      - tipo: str ('fijo', 'mercado_favorito', 'cercano', 'carniceria')
      - carnicerias_abiertas: bool
      - cercanos_abiertos: bool
      - sostiene_precio: bool
      - precio_mas_barato: bool
    """
    pass

class MarcoOperacion(Fact):
    """
    Marco de Conocimiento para Estados del Servicio y Tareas Operativas.
    Ranuras (Slots):
      - tarea: str ('compra_diaria', 'ingreso_mercaderia', 'llegada_mercaderia')
      - comanda: str ('carne', 'sandwiches', 'bebidas')
      - duena_disponible: bool (False activa delegación)
      - quiebre_stock_critico: bool (True dispara protocolo de crisis)
    """
    pass

class HechoSimbolico(Fact):
    """
    Representación Híbrida / Simbólica Directa.
    Permite declarar y equiparar proposiciones mediante su letra (A - BN).
    """
    pass

# Alias convenientes para el equipo
Contexto = MarcoContexto
Cliente = MarcoCliente
Insumo = MarcoInsumo
Proveedor = MarcoProveedor
Operacion = MarcoOperacion

print("✅ Marcos de Conocimiento (Frames) definidos correctamente.")


✅ Marcos de Conocimiento (Frames) definidos correctamente.


---
## 3. Diccionario Canónico de Proposiciones (A - BN)

Estructura canónica de referencia con las 66 proposiciones formalizadas en la Actividad 4, categorizadas y mapeadas a los atributos de los Marcos:


In [ ]:
DICCIONARIO_PROPOSICIONES = {
    # Módulo A: Demanda y Menú
    "A": {"desc": "Día es Fin de semana", "categoria": "Contexto", "ranura": "dia", "valor": "fin_de_semana"},
    "B": {"desc": "Día es Domingo", "categoria": "Contexto", "ranura": "dia", "valor": "domingo"},
    "C": {"desc": "Aumentar compra de mercadería", "categoria": "Accion", "ranura": "compra", "valor": "aumentar"},
    "D": {"desc": "Surtir para mañana, tarde y noche", "categoria": "Accion", "ranura": "turnos_abastecimiento", "valor": "todos"},
    "E": {"desc": "Evento es Partido Boca/River", "categoria": "Contexto", "ranura": "evento", "valor": "boca_river"},
    "F": {"desc": "Planificar turnos temprano y tarde", "categoria": "Accion", "ranura": "turnos_personal", "valor": "temprano_y_tarde"},
    "G": {"desc": "Comprar bebidas de forma gradual", "categoria": "Accion", "ranura": "estrategia_bebida", "valor": "gradual"},
    "H": {"desc": "No comprar bebidas de golpe", "categoria": "Accion", "ranura": "estrategia_bebida", "valor": "no_de_golpe"},
    "I": {"desc": "Evento es Desfile 23 de Agosto", "categoria": "Contexto", "ranura": "evento", "valor": "desfile_23_agosto"},
    "J": {"desc": "Competencia es Alta", "categoria": "Contexto", "ranura": "competencia", "valor": "alta"},
    "K": {"desc": "Reducir expectativa de demanda", "categoria": "Accion", "ranura": "expectativa_demanda", "valor": "reducida"},
    "L": {"desc": "Evitar sobrestock", "categoria": "Accion", "ranura": "riesgo_stock", "valor": "evitar_sobrestock"},
    "M": {"desc": "Temperatura es frío", "categoria": "Contexto", "ranura": "temperatura", "valor": "frio"},
    "N": {"desc": "Ofrecer menú caliente (sopas)", "categoria": "Accion", "ranura": "oferta_menu", "valor": "caliente_sopas"},
    "O": {"desc": "Mantener menú habitual", "categoria": "Accion", "ranura": "oferta_menu", "valor": "mantener_habitual"},
    "P": {"desc": "Ofrecer sándwiches y pizzas", "categoria": "Accion", "ranura": "oferta_menu", "valor": "sandwiches_pizzas"},
    "Q": {"desc": "Fecha es quincena o cobro", "categoria": "Contexto", "ranura": "es_quincena", "valor": True},
    "R": {"desc": "Clima es soleado", "categoria": "Contexto", "ranura": "clima", "valor": "soleado"},
    "S": {"desc": "Reservas son grupos grandes", "categoria": "Cliente", "ranura": "reservas", "valor": "grupos_grandes"},
    "T": {"desc": "Comportamiento es Alta Bebida y Baja Comida", "categoria": "Cliente", "ranura": "comportamiento", "valor": "alta_bebida_baja_comida"},
    "U": {"desc": "Aumentar compra de destilados y snacks", "categoria": "Accion", "ranura": "compras_especificas", "valor": "mas_destilados_snacks"},
    "V": {"desc": "Reducir compra de vegetales", "categoria": "Accion", "ranura": "compras_especificas", "valor": "menos_vegetales"},
    "W": {"desc": "Tipo de cliente es Familia", "categoria": "Cliente", "ranura": "tipo", "valor": "familia"},
    "X": {"desc": "Preferencia es Menú al plato", "categoria": "Cliente", "ranura": "preferencia", "valor": "menu_al_plato"},
    "Y": {"desc": "Tipo de cliente es Jóvenes", "categoria": "Cliente", "ranura": "tipo", "valor": "jovenes"},
    "Z": {"desc": "Preferencia es Bebidas y sándwiches", "categoria": "Cliente", "ranura": "preferencia", "valor": "bebidas_y_sandwiches"},

    # Módulo B: Compras y Prioridades
    "AA": {"desc": "Producto es Cerveza", "categoria": "Insumo", "ranura": "nombre", "valor": "cerveza"},
    "AB": {"desc": "Prioridad de compra es Alta y Rotación muy rápida", "categoria": "Accion", "ranura": "prioridad_rotacion", "valor": "alta_muy_rapida"},
    "AC": {"desc": "Producto es Pre-pizza, Queso, Jamón o Salsa", "categoria": "Insumo", "ranura": "categoria", "valor": "pizza_y_aderezos"},
    "AD": {"desc": "Prioridad de compra es Alta y Rotación rápida", "categoria": "Accion", "ranura": "prioridad_rotacion", "valor": "alta_rapida"},
    "AE": {"desc": "Tarea es Compra Diaria", "categoria": "Operacion", "ranura": "tarea", "valor": "compra_diaria"},
    "AF": {"desc": "Desplazarse a mercados favoritos", "categoria": "Accion", "ranura": "metodo_compra", "valor": "mercados_favoritos"},
    "AG": {"desc": "Pagar con Efectivo, Transferencia o Tarjeta", "categoria": "Accion", "ranura": "medios_pago", "valor": "efectivo_transferencia_tarjeta"},
    "AH": {"desc": "Dueña no está disponible", "categoria": "Operacion", "ranura": "duena_disponible", "valor": False},
    "AI": {"desc": "Delegar compra a Encargada de caja", "categoria": "Accion", "ranura": "delegacion", "valor": "encargada_caja"},
    "AJ": {"desc": "Producto es Hielo", "categoria": "Insumo", "ranura": "nombre", "valor": "hielo"},
    "AK": {"desc": "Origen de compra son Proveedores Fijos", "categoria": "Accion", "ranura": "origen_compra", "valor": "proveedores_fijos"},
    "AL": {"desc": "Local favorito no sostiene precio", "categoria": "Proveedor", "ranura": "sostiene_precio", "valor": False},
    "AM": {"desc": "Otro local tiene precio más barato", "categoria": "Proveedor", "ranura": "precio_mas_barato", "valor": True},
    "AN": {"desc": "Cambiar de local/proveedor", "categoria": "Accion", "ranura": "decision_proveedor", "valor": "cambiar_proveedor"},

    # Módulo C: Conservación y Almacenamiento
    "AO": {"desc": "Producto es Bebida Blanca, Gaseosa o Vino", "categoria": "Insumo", "ranura": "categoria", "valor": "bebidas_generales"},
    "AP": {"desc": "Destino de almacenamiento es Depósito", "categoria": "Accion", "ranura": "almacenamiento", "valor": "deposito"},
    "AQ": {"desc": "Ingresa mercadería nueva", "categoria": "Operacion", "ranura": "tarea", "valor": "ingreso_mercaderia"},
    "AR": {"desc": "Ubicar nuevo atrás y viejo adelante (FIFO / PEPS)", "categoria": "Accion", "ranura": "criterio_estiba", "valor": "peps_fifo"},
    "AS": {"desc": "Producto es Carne o Comida Diaria", "categoria": "Insumo", "ranura": "categoria", "valor": "perecedero_diario"},
    "AT": {"desc": "Tiempo de guardado corto y compra diaria", "categoria": "Accion", "ranura": "politica_stock", "valor": "guardado_corto_diario"},
    "AU": {"desc": "Producto es Carne", "categoria": "Insumo", "ranura": "nombre", "valor": "carne"},
    "AV": {"desc": "Separar en porciones con folex", "categoria": "Accion", "ranura": "manipulacion", "valor": "fraccionar_con_folex"},
    "AW": {"desc": "Producto es Carne en folex", "categoria": "Insumo", "ranura": "estado", "valor": "fraccionada_en_folex"},
    "AX": {"desc": "Congelar", "categoria": "Accion", "ranura": "conservacion", "valor": "congelar"},
    "AY": {"desc": "Ingresa comanda de Carne", "categoria": "Operacion", "ranura": "comanda", "valor": "carne"},
    "AZ": {"desc": "Sacar porción congelada directo a plancha", "categoria": "Accion", "ranura": "coccion", "valor": "directo_a_plancha"},
    "BA": {"desc": "Llega mercadería al local", "categoria": "Operacion", "ranura": "tarea", "valor": "llegada_mercaderia"},
    "BB": {"desc": "Acomodar lugares y conservar para horas pico", "categoria": "Accion", "ranura": "operacion_salon", "valor": "preparar_para_pico"},

    # Módulo D: Control de Crisis Operativas
    "BC": {"desc": "Momento es Crítico y Falta Insumo", "categoria": "Operacion", "ranura": "quiebre_stock_critico", "valor": True},
    "BD": {"desc": "Negocios cercanos abiertos", "categoria": "Proveedor", "ranura": "cercanos_abiertos", "valor": True},
    "BE": {"desc": "Mandar empleado a comprar", "categoria": "Accion", "ranura": "gestion_crisis", "valor": "comprar_urgencia_empleado"},
    "BF": {"desc": "Día es Domingo por la tarde", "categoria": "Contexto", "ranura": "horario_critico", "valor": "domingo_tarde"},
    "BG": {"desc": "Carnicerías están cerradas", "categoria": "Proveedor", "ranura": "carnicerias_abiertas", "valor": False},
    "BH": {"desc": "Dar de baja producto y ofrecer alternativa", "categoria": "Accion", "ranura": "gestion_crisis", "valor": "baja_producto_y_alternativa"},

    # Módulo E: Mermas y Ajuste de Precios
    "BI": {"desc": "Color de carne es oscuro", "categoria": "Insumo", "ranura": "apariencia_carne", "valor": "color_oscuro"},
    "BJ": {"desc": "Separar de venta y cocinar para animales de la calle", "categoria": "Accion", "ranura": "destino_merma", "valor": "separar_y_cocinar_animales"},
    "BK": {"desc": "Aumenta costo de carne", "categoria": "Insumo", "ranura": "aumento_costo_carne", "valor": True},
    "BL": {"desc": "Aumenta precio de plato", "categoria": "Accion", "ranura": "ajuste_precio", "valor": "trasladar_al_plato"},
    "BM": {"desc": "Aumenta costo de bebida", "categoria": "Insumo", "ranura": "aumento_costo_bebida", "valor": True},
    "BN": {"desc": "Mantener precio de bebida", "categoria": "Accion", "ranura": "ajuste_precio", "valor": "mantener_precio_bebida"}
}

def obtener_hechos_simbolicos(*codigos):
    return [HechoSimbolico(codigo=c, desc=DICCIONARIO_PROPOSICIONES[c]['desc'], valor=True) for c in codigos if c in DICCIONARIO_PROPOSICIONES]

print(f"✅ Diccionario cargado con {len(DICCIONARIO_PROPOSICIONES)} proposiciones.")


✅ Diccionario cargado con 66 proposiciones.


---
## 4. Estructuración y Carga de la Memoria de Trabajo Inicial (WM0)

Funciones de factoría para cargar los escenarios de la Actividad 4:
* **Caso 1:** Ajuste de Menú por Clima y Público ($M, W \implies N, O, P, X$)
* **Caso 2:** Alta Demanda por Quincena y Reservas ($Q, R, S \implies T, U, V$)
* **Caso 3:** Crisis de Abastecimiento Domingo Tarde ($BC, BF, BG \implies BH$)
* **Carga Interactiva:** Permite al usuario/docente definir condiciones del día en vivo.


In [ ]:
def cargar_caso_1():
    print("📋 Cargando Escenario: CASO 1 (Clima frío y público familiar)")
    return [
        Contexto(temperatura="frio"),
        Cliente(tipo="familia")
    ] + obtener_hechos_simbolicos("M", "W")

def cargar_caso_2():
    print("📋 Cargando Escenario: CASO 2 (Quincena, sol y grupos grandes)")
    return [
        Contexto(es_quincena=True, clima="soleado"),
        Cliente(reservas="grupos_grandes")
    ] + obtener_hechos_simbolicos("Q", "R", "S")

def cargar_caso_3():
    print("📋 Cargando Escenario: CASO 3 (Faltante crítico domingo tarde con comercios cerrados)")
    return [
        Operacion(quiebre_stock_critico=True),
        Contexto(dia="domingo", momento="tarde", horario_critico="domingo_tarde"),
        Proveedor(carnicerias_abiertas=False)
    ] + obtener_hechos_simbolicos("BC", "BF", "BG")

def cargar_interactivo():
    print("=== Configuración Interactiva de Hechos Iniciales ===")
    dia = input("Día [domingo / fin_de_semana / habil]: ") or "domingo"
    temp = input("Temperatura [frio / templado / calido]: ") or "frio"
    cliente = input("Tipo de cliente [familia / jovenes]: ") or "familia"
    return [
        Contexto(dia=dia, temperatura=temp),
        Cliente(tipo=cliente)
    ]


---
## 5. Validación y Demostración de Encadenamiento Hacia Adelante (Caso 1)

Verificamos que los Marcos de Conocimiento se equiparan adecuadamente con el motor de inferencia de Experta:


In [ ]:
class MotorPruebaPunto2(KnowledgeEngine):
    """
    Motor mínimo de verificación para validar la Base de Hechos del Punto 2.
    """

    # Regla 5: Clima frío -> Ofrecer menú caliente y sopas
    @Rule(
        OR(
            Contexto(temperatura="frio"),
            HechoSimbolico(codigo="M")
        ),
        NOT(HechoSimbolico(codigo="N"))
    )
    def regla_5_clima_frio_sopas(self):
        print(" -> [DISPARO R5]: Temperatura fría detectada.")
        print("    Inferido Hecho N: Ofrecer menú caliente (sopas).")
        self.declare(HechoSimbolico(codigo="N", desc="Ofrecer menú caliente (sopas)", valor=True))

    # Regla 6: Clima frío -> Mantener menú habitual (sándwiches y pizzas)
    @Rule(
        OR(
            Contexto(temperatura="frio"),
            HechoSimbolico(codigo="M")
        ),
        NOT(HechoSimbolico(codigo="O"))
    )
    def regla_6_clima_frio_sandwiches(self):
        print(" -> [DISPARO R6]: Temperatura fría detectada.")
        print("    Inferidos Hechos O y P: Mantener habitual y ofrecer sándwiches/pizzas.")
        self.declare(HechoSimbolico(codigo="O", desc="Mantener menú habitual", valor=True))
        self.declare(HechoSimbolico(codigo="P", desc="Ofrecer sándwiches y pizzas", valor=True))

    # Regla 10: Cliente Familia -> Preferencia Menú al plato
    @Rule(
        OR(
            Cliente(tipo="familia"),
            HechoSimbolico(codigo="W")
        ),
        NOT(HechoSimbolico(codigo="X"))
    )
    def regla_10_cliente_familia(self):
        print(" -> [DISPARO R10]: Perfil de cliente Familia detectado.")
        print("    Inferido Hecho X: Preferencia por Menú al plato.")
        self.declare(Cliente(preferencia="menu_al_plato"))
        self.declare(HechoSimbolico(codigo="X", desc="Preferencia es Menú al plato", valor=True))

# Ejecución del Caso 1
motor = MotorPruebaPunto2()
motor.reset()

hechos_caso1 = cargar_caso_1()
for hecho in hechos_caso1:
    motor.declare(hecho)

print("\n🚀 Disparando Motor de Inferencia (Encadenamiento Hacia Adelante)...")
motor.run()
print("\n🏁 Encadenamiento finalizado con éxito.")


📋 Cargando Escenario: CASO 1 (Clima frío y público familiar)

🚀 Disparando Motor de Inferencia (Encadenamiento Hacia Adelante)...
 -> [DISPARO R10]: Perfil de cliente Familia detectado.
    Inferido Hecho X: Preferencia por Menú al plato.
 -> [DISPARO R5]: Temperatura fría detectada.
    Inferido Hecho N: Ofrecer menú caliente (sopas).
 -> [DISPARO R6]: Temperatura fría detectada.
    Inferidos Hechos O y P: Mantener habitual y ofrecer sándwiches/pizzas.
 -> [DISPARO R10]: Perfil de cliente Familia detectado.
    Inferido Hecho X: Preferencia por Menú al plato.
 -> [DISPARO R5]: Temperatura fría detectada.
    Inferido Hecho N: Ofrecer menú caliente (sopas).
 -> [DISPARO R6]: Temperatura fría detectada.
    Inferidos Hechos O y P: Mantener habitual y ofrecer sándwiches/pizzas.

🏁 Encadenamiento finalizado con éxito.


🧠 Trabajo de Campo – Inteligencia del Conocimiento (InCo)
Sistema Basado en Conocimiento: Optimización de Inventarios
📌 Punto 4: Ingeniero de Reglas Lógicas 2 (Módulos C, D y E)

In [9]:
"""
Módulo de Reglas Lógicas: Módulos C, D y E
Trabajo de Campo - Inteligencia del Conocimiento (InCo 2026)
Punto 4: Ingeniero de Reglas Lógicas 2 (Desarrollado por Jonatan Perez Apumaita / Auditado y blindado contra bucles y deadlocks)

Implementa:
- Módulo C: Almacenamiento y Conservación (R19 a R25)
- Módulo D: Operación en Servicio y Control de Crisis (R26 y R27)
- Módulo E: Control de Mermas y Ajuste de Precios (R28 a R30)

Garantías de Diseño (Auditoría Anti-Bucles y Anti-Deadlocks):
1. Todas las reglas cuentan con guardas `NOT(HechoSimbolico(...))` para evitar dobles disparos o activaciones redundantes.
2. El encadenamiento de conservación de carne (R22 -> R23 -> R24) asevera el estado intermedio `AW` para evitar bloqueos.
3. Las reglas de crisis y seguridad bromatológica (R27 y R28) cuentan con salience prioritario para resolver conflictos determinísticamente.
"""

from marcos import (
    KnowledgeEngine, Rule, Fact, Contexto, Cliente, Insumo,
    Proveedor, Operacion, HechoSimbolico, OR, AND, NOT,
    cargar_caso_3
)

class MotorModulosCDE(KnowledgeEngine):
    """
    Motor de Inferencia para los Módulos C, D y E.
    Blindado contra bucles infinitos, dobles disparos y deadlocks.
    """

    # --- MÓDULO C: Almacenamiento y Conservación ---
    @Rule(
        OR(Insumo(categoria="bebidas_generales"), HechoSimbolico(codigo="AO")),
        NOT(HechoSimbolico(codigo="AP")),
        salience=10
    )
    def r19_almacenamiento_bebidas(self):
        self.declare(HechoSimbolico(codigo="AP", desc="Destino a Depósito", valor=True))
        print(" -> [FIRE R19] Almacenamiento: Bebidas enviadas a depósito.")

    @Rule(
        OR(Operacion(tarea="ingreso_mercaderia"), HechoSimbolico(codigo="AQ")),
        NOT(HechoSimbolico(codigo="AR")),
        salience=10
    )
    def r20_estiba_peps(self):
        self.declare(HechoSimbolico(codigo="AR", desc="Estiba PEPS", valor=True))
        print(" -> [FIRE R20] Estiba: Acomodando mercadería nueva atrás y vieja adelante (PEPS/FIFO).")

    @Rule(
        OR(Insumo(categoria="perecedero_diario"), HechoSimbolico(codigo="AS")),
        NOT(HechoSimbolico(codigo="AT")),
        salience=10
    )
    def r21_tiempo_guardado(self):
        self.declare(HechoSimbolico(codigo="AT", desc="Guardado corto", valor=True))
        print(" -> [FIRE R21] Stock: Tiempo de guardado corto y compra diaria.")

    @Rule(
        OR(Insumo(nombre="carne"), HechoSimbolico(codigo="AU")),
        NOT(HechoSimbolico(codigo="AV")),
        salience=10
    )
    def r22_fraccionar_carne(self):
        print(" -> [FIRE R22] Carnicería: Separando carne en porciones con folex.")
        self.declare(HechoSimbolico(codigo="AV", desc="Fraccionar en folex", valor=True))
        # Aseverar el estado intermedio AW para habilitar el encadenamiento hacia R23 sin deadlock
        self.declare(HechoSimbolico(codigo="AW", desc="Producto es Carne en folex", valor=True))
        self.declare(Insumo(estado="fraccionada_en_folex"))

    @Rule(
        OR(Insumo(estado="fraccionada_en_folex"), HechoSimbolico(codigo="AW")),
        NOT(HechoSimbolico(codigo="AX")),
        salience=9
    )
    def r23_congelar_carne(self):
        print(" -> [FIRE R23] Conservación: Carne en folex detectada. Congelar de inmediato.")
        self.declare(HechoSimbolico(codigo="AX", desc="Congelar", valor=True))

    @Rule(
        OR(Operacion(comanda="carne"), HechoSimbolico(codigo="AY")),
        NOT(HechoSimbolico(codigo="AZ")),
        salience=10
    )
    def r24_coccion_carne(self):
        print(" -> [FIRE R24] Cocina: Comanda de carne ingresada. Sacar porción directo a la plancha.")
        self.declare(HechoSimbolico(codigo="AZ", desc="Carne a la plancha", valor=True))

    @Rule(
        OR(Operacion(tarea="llegada_mercaderia"), HechoSimbolico(codigo="BA")),
        NOT(HechoSimbolico(codigo="BB")),
        salience=10
    )
    def r25_acomodar_salon(self):
        print(" -> [FIRE R25] Salón: Mercadería en local. Acomodar lugares y conservar para horas pico.")
        self.declare(HechoSimbolico(codigo="BB", desc="Conservar horas pico", valor=True))

    # --- MÓDULO D: Operación en Servicio (Crisis y Faltantes) ---
    @Rule(
        AND(
            OR(Operacion(quiebre_stock_critico=True), HechoSimbolico(codigo="BC")),
            OR(Proveedor(cercanos_abiertos=True), HechoSimbolico(codigo="BD"))
        ),
        NOT(HechoSimbolico(codigo="BE")),
        salience=10
    )
    def r26_comprar_urgencia(self):
        self.declare(HechoSimbolico(codigo="BE", desc="Compra de urgencia", valor=True))
        print(" -> [FIRE R26] Crisis: Comercios cercanos abiertos. Mandando empleado a comprar urgente.")

    @Rule(
        AND(
            OR(Operacion(quiebre_stock_critico=True), HechoSimbolico(codigo="BC")),
            OR(Contexto(horario_critico="domingo_tarde"), HechoSimbolico(codigo="BF")),
            OR(Proveedor(carnicerias_abiertas=False), HechoSimbolico(codigo="BG"))
        ),
        NOT(HechoSimbolico(codigo="BH")),
        salience=15 # Salience prioritario sobre R26 ante falta de comercios especializados
    )
    def r27_baja_producto(self):
        self.declare(HechoSimbolico(codigo="BH", desc="Baja de producto", valor=True))
        print(" 🚨 [FIRE R27] Crisis: Domingo tarde y carnicerías cerradas. Dar de baja el producto y ofrecer alternativa.")

    # --- MÓDULO E: Control de Mermas y Ajuste de Precios ---
    @Rule(
        OR(Insumo(apariencia_carne="color_oscuro"), HechoSimbolico(codigo="BI")),
        NOT(HechoSimbolico(codigo="BJ")),
        salience=20 # Máxima prioridad: faceta procedimental de inocuidad alimentaria
    )
    def r28_merma_carne(self):
        self.declare(HechoSimbolico(codigo="BJ", desc="Separar venta", valor=True))
        print(" ⚠️ [FIRE R28] Merma: Carne oscura separada de venta para cocinar a animales de la calle.")

    @Rule(
        OR(Insumo(aumento_costo_carne=True), HechoSimbolico(codigo="BK")),
        NOT(HechoSimbolico(codigo="BL")),
        salience=10
    )
    def r29_aumento_plato(self):
        self.declare(HechoSimbolico(codigo="BL", desc="Aumento de plato", valor=True))
        print(" -> [FIRE R29] Precios: Aumenta costo de carne, trasladar incremento al precio del plato.")

    @Rule(
        OR(Insumo(aumento_costo_bebida=True), HechoSimbolico(codigo="BM")),
        NOT(HechoSimbolico(codigo="BN")),
        salience=10
    )
    def r30_mantiene_bebida(self):
        self.declare(HechoSimbolico(codigo="BN", desc="Mantener bebida", valor=True))
        print(" -> [FIRE R30] Precios: Aumenta costo de bebida, mantener precio de venta para fidelización.")

if __name__ == "__main__":
    print("\n" + "="*65)
    print("TEST DE INTEGRACIÓN: MÓDULOS C, D y E (CASO 3 AUDITADO)")
    print("="*65)
    motor_jona = MotorModulosCDE()
    motor_jona.reset()

    hechos_caso3 = cargar_caso_3()
    for hecho in hechos_caso3:
        motor_jona.declare(hecho)

    print("\n🚀 Ejecutando el motor para el Caso 3 (Forward Chaining)...\n")
    motor_jona.run()
    print("\n🏁 Fin de la ejecución de los módulos C, D y E sin bucles ni deadlocks.")



--- TEST DE INTEGRACIÓN: MÓDULOS C, D y E con CASO 3 ---
📋 Cargando Escenario: CASO 3 (Faltante crítico domingo tarde con comercios cerrados)

🚀 Ejecutando el motor para el Caso 3 (Forward Chaining)...

 -> [R27] Crisis: Carnicerías cerradas. Dar de baja el producto y ofrecer alternativa.

🏁 Fin de la ejecución de los módulos C, D y E.
